# GenoMed AI: explanation module (Colab)

Takes one structured genetic result (`INPUT`) and writes two summaries with Claude:

* `clinician_summary`: 3 to 5 bullets for the doctor
* `patient_summary` and `patient_questions`: plain language, for the doctor to review before sending

This is a **decision-support prototype**. It does not diagnose, classify variants or recommend treatment.

**How the notebook works**
1. Checks the input in code (gene, variant and `clinvar.classification` present, no obvious contradictions). Bad input returns an `error` without calling the model.
2. Calls Claude with your system prompt and a JSON schema, so the reply is always valid JSON in the required shape.
3. Runs guardrail checks on the output: banned phrases, numbers not found in `INPUT`, AI or model mentions in the patient text, word and bullet counts. Problems are listed as `warnings` for the reviewing doctor. The output is never silently changed.

**Setup:** in Colab, open the 🔑 **Secrets** panel (left sidebar), add `ANTHROPIC_API_KEY`, and turn on notebook access.

In [ ]:
%pip install -q -U "anthropic>=2.14"

In [ ]:
import os, json, re
import anthropic

# Read the API key from Colab Secrets (falls back to an env var outside Colab).
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except ImportError:
    pass  # not in Colab: expects ANTHROPIC_API_KEY in the environment

client = anthropic.Anthropic()
MODEL = "claude-opus-5-5"

## 1. System prompt
Your prompt, unchanged.

In [ ]:
SYSTEM_PROMPT = """You are the explanation module of GenoMed AI, a decision-support prototype that helps doctors explain genetic test results. You do not diagnose, classify variants or recommend treatment. You rewrite the structured result you are given into two short summaries.

You receive one JSON object called INPUT. Use only the facts in INPUT: gene_info, clinvar, ai_estimate and context_facts. Do not add any other medical fact, statistic, risk percentage, treatment or test name, even if you believe it is true. If a fact you would need is missing, leave it out.

Write in the language given by INPUT.language ("en" for English, "ka" for Georgian). Use natural, everyday wording, not a word-for-word translation.

Produce two summaries.

1. clinician_summary (for the doctor)
- 3 to 5 bullet points, each under 25 words.
- State the ClinVar classification and its review status.
- State the AI estimate label and that it is research-grade and not clinically validated.
- If the AI estimate and the ClinVar classification seem to disagree, say so plainly. Never present the AI estimate as overriding ClinVar.
- End with next steps drawn only from context_facts, phrased as considerations, not orders.

2. patient_summary (for the patient, reviewed by the doctor before sending)
- 80 to 150 words, short sentences, words a 12-year-old understands. Explain any medical word the first time you use it.
- Explain what the gene does using gene_info.
- Explain what the result means using the clinvar classification and context_facts.
- Never mention the AI estimate, models, scores or Evo 2.
- Calm and honest. Do not frighten, and do not falsely reassure.
- Then give 2 to 4 questions the patient can ask their doctor, in a separate list.

Never write any of the following in either summary:
- that the patient has, will get or will not get a disease;
- "you are safe", "nothing to worry about", "don't worry" or similar;
- any recommendation to have or avoid surgery, medication or treatment;
- any number or percentage that is not in INPUT.

If INPUT is missing gene, variant or clinvar.classification, or the values contradict each other, do not write summaries. Return only the error field explaining what is missing.

Return only valid JSON in exactly this shape, with no text before or after it:
{
  "clinician_summary": ["...", "..."],
  "patient_summary": "...",
  "patient_questions": ["...", "..."],
  "error": null
}"""

## 2. Output schema
Structured outputs force the reply into this shape. In the error case the three summary fields are `null` and only `error` is filled.

In [ ]:
def _nullable(schema):
    return {"anyOf": [schema, {"type": "null"}]}

OUTPUT_SCHEMA = {
    "type": "object",
    "properties": {
        "clinician_summary": _nullable({"type": "array", "items": {"type": "string"}}),
        "patient_summary":   _nullable({"type": "string"}),
        "patient_questions": _nullable({"type": "array", "items": {"type": "string"}}),
        "error":             _nullable({"type": "string"}),
    },
    "required": ["clinician_summary", "patient_summary", "patient_questions", "error"],
    "additionalProperties": False,
}

def error_result(msg):
    return {"clinician_summary": None, "patient_summary": None, "patient_questions": None, "error": msg}

## 3. Input checks (in code, before the model)
Missing required fields are caught here, so a bad record never reaches the model. Subtler contradictions are still left to the model, as the prompt says.

In [ ]:
def check_input(inp):
    """Returns an error message, or None if the input is usable."""
    if not isinstance(inp, dict):
        return "INPUT is not a JSON object."
    missing = []
    if not str(inp.get("gene") or "").strip():
        missing.append("gene")
    if not str(inp.get("variant") or "").strip():
        missing.append("variant")
    clinvar = inp.get("clinvar") or {}
    if not str(clinvar.get("classification") or "").strip():
        missing.append("clinvar.classification")
    if missing:
        return "INPUT is missing: " + ", ".join(missing) + "."

    if inp.get("language") not in ("en", "ka"):
        return 'INPUT.language must be "en" or "ka".'

    # Contradictions we can detect mechanically.
    gi = inp.get("gene_info") or {}
    sym = gi.get("symbol") or gi.get("gene")
    if sym and str(sym).strip().upper() != str(inp["gene"]).strip().upper():
        return f'Contradiction: gene is "{inp["gene"]}" but gene_info describes "{sym}".'
    for key in ("gene", "variant"):
        v = clinvar.get(key)
        if v and str(v).strip().upper() != str(inp[key]).strip().upper():
            return f'Contradiction: INPUT.{key} is "{inp[key]}" but clinvar.{key} is "{v}".'
    return None

## 4. Output guardrails (in code, after the model)
These do **not** change the text. They list possible problems for the reviewing doctor.
The Georgian phrase list is a starting point and should be checked and extended by a native speaker.

In [ ]:
BANNED_PHRASES = {
    "en": ["you are safe", "you're safe", "nothing to worry", "don't worry", "do not worry",
           "no need to worry", "no cause for concern", "you will get", "you will not get",
           "you won't get", "you have cancer", "you have the disease", "you should have surgery",
           "you should take", "you should start", "you should stop", "you should avoid"],
    "ka": ["არ ინერვიულოთ", "არ ინერვიულო", "უსაფრთხოდ ხართ", "უსაფრთხო ხართ", "საშიში არაფერია",
           "სანერვიულო არაფერია"],
}
AI_TERMS = [r"\bai\b", r"a\.i\.", "artificial intelligence", r"\bmodels?\b", r"evo ?2", r"\bscores?\b",
            "algorithm", "machine learning", "ხელოვნური ინტელექტ", "მოდელ", "ქულა", "ალგორითმ"]
# "conflict" alone is not used: ClinVar review statuses often say "no conflicts".
DISAGREE_TERMS = ["disagree", "differ", "conflicts with", "in conflict", "discordan", "contradict",
                  "inconsistent", "does not match", "do not match",
                  "განსხვავ", "ეწინააღმდეგ", "არ ემთხვევა"]

def _numbers(text):
    return set(re.findall(r"\d+(?:[.,]\d+)?", text))

def _words(text):
    return len(re.findall(r"\S+", text))

def _bucket(label):
    s = (label or "").lower()
    if "benign" in s or "კეთილთვისებ" in s:
        return "benign"
    if "pathogenic" in s or "პათოგენ" in s:
        return "pathogenic"
    return "uncertain"

def check_output(inp, out):
    warnings = []
    if out.get("error"):
        return warnings
    clin = out.get("clinician_summary") or []
    pat = out.get("patient_summary") or ""
    qs = out.get("patient_questions") or []
    lang = inp.get("language", "en")

    if not 3 <= len(clin) <= 5:
        warnings.append(f"clinician_summary has {len(clin)} bullets (expected 3-5).")
    for i, b in enumerate(clin, 1):
        if _words(b) >= 25:
            warnings.append(f"clinician bullet {i} has {_words(b)} words (expected under 25).")
    n = _words(pat)
    if not 80 <= n <= 150:
        warnings.append(f"patient_summary has {n} words (expected 80-150).")
    if not 2 <= len(qs) <= 4:
        warnings.append(f"patient_questions has {len(qs)} items (expected 2-4).")

    all_text = " ".join(clin + [pat] + qs)
    low = all_text.lower()
    for p in dict.fromkeys(BANNED_PHRASES.get(lang, []) + BANNED_PHRASES["en"]):
        if p.lower() in low:
            warnings.append(f'Banned phrase found: "{p}".')

    patient_low = (pat + " " + " ".join(qs)).lower()
    for t in AI_TERMS:
        m = re.search(t, patient_low)
        if m:
            warnings.append(f'Patient text may mention the AI estimate (found "{m.group(0)}").')

    allowed = _numbers(json.dumps(inp, ensure_ascii=False))
    extra = sorted(_numbers(all_text) - allowed)
    if extra:
        warnings.append(f"Numbers not found in INPUT: {', '.join(extra)}.")
    if "%" in all_text and "%" not in json.dumps(inp):
        warnings.append("A percentage sign appears but INPUT has no percentages.")

    ai_label = (inp.get("ai_estimate") or {}).get("label")
    a, c = _bucket(ai_label), _bucket(inp["clinvar"]["classification"])
    if ai_label and a != c and not any(t in " ".join(clin).lower() for t in DISAGREE_TERMS):
        warnings.append(f"AI estimate ({ai_label}) and ClinVar ({inp['clinvar']['classification']}) "
                        "seem to disagree, but the clinician summary does not say so.")
    return warnings

## 5. Explain function
`fallbacks="default"` is turned on: if Claude's safety filter declines a request (rare, but genetics content can trigger it), the API retries on a fallback model in the same call. Remove the `betas` and `fallbacks` lines to turn this off.

In [ ]:
def explain(inp, effort="medium"):
    err = check_input(inp)
    if err:
        return error_result(err)

    try:
        response = client.beta.messages.create(
            model=MODEL,
            max_tokens=16000,
            system=SYSTEM_PROMPT,
            messages=[{"role": "user",
                       "content": "INPUT:\n" + json.dumps(inp, ensure_ascii=False, indent=2)}],
            output_config={"effort": effort,
                           "format": {"type": "json_schema", "schema": OUTPUT_SCHEMA}},
            betas=["server-side-fallback-2026-07-01"],
            fallbacks="default",
        )
    except anthropic.AuthenticationError:
        return error_result("API key missing or invalid. Add ANTHROPIC_API_KEY in Colab Secrets.")
    except anthropic.BadRequestError as e:
        return error_result(f"Bad request: {e.message}")
    except anthropic.RateLimitError:
        return error_result("Rate limited. Wait a minute and run again.")
    except anthropic.APIStatusError as e:
        return error_result(f"API error {e.status_code}: {e.message}")
    except anthropic.APIConnectionError:
        return error_result("Could not reach the Claude API.")

    if response.stop_reason == "refusal":
        cat = response.stop_details.category if response.stop_details else None
        return error_result(f"The model declined this request (category: {cat}).")
    if response.stop_reason == "max_tokens":
        return error_result("Output was cut off (max_tokens).")

    # Use the text after the last fallback switch, if any.
    blocks = list(response.content)
    last_fb = max((i for i, b in enumerate(blocks) if b.type == "fallback"), default=-1)
    text = "".join(b.text for b in blocks[last_fb + 1:] if b.type == "text")
    out = json.loads(text)

    if out.get("error"):
        out = error_result(out["error"])
    out["warnings"] = check_output(inp, out)
    out["_served_by"] = response.model
    return out

def show(out):
    print(json.dumps(out, ensure_ascii=False, indent=2))

## 6. Demo inputs
⚠️ **Made-up demo records** to show the format. They are not real clinical data. Replace them with your pipeline's output.

In [ ]:
DEMO_AGREE = {
    "language": "en",
    "gene": "GENEX",
    "variant": "c.100A>G (p.Lys34Glu)",
    "gene_info": {
        "symbol": "GENEX",
        "function": "Makes a protein that helps the body repair damaged DNA in cells."
    },
    "clinvar": {
        "classification": "Uncertain significance",
        "review_status": "criteria provided, multiple submitters, no conflicts"
    },
    "ai_estimate": {"label": "Uncertain", "model": "Evo 2", "score": -0.41},
    "context_facts": [
        "A variant of uncertain significance means there is not enough evidence to say if it affects health.",
        "Classifications can change as more evidence is collected.",
        "Family history can help the care team interpret the result.",
        "A genetic counselor can explain the result in more detail."
    ]
}

# Same record, but the AI estimate disagrees with ClinVar.
DEMO_DISAGREE = {**DEMO_AGREE,
                 "ai_estimate": {"label": "Likely pathogenic", "model": "Evo 2", "score": -2.7}}

# Georgian output.
DEMO_KA = {**DEMO_AGREE, "language": "ka"}

# Missing clinvar.classification: should return an error without calling the model.
DEMO_MISSING = {**DEMO_AGREE, "clinvar": {"review_status": "no assertion criteria provided"}}

In [ ]:
show(explain(DEMO_MISSING))   # no API call

In [ ]:
show(explain(DEMO_AGREE))

In [ ]:
show(explain(DEMO_DISAGREE))

In [ ]:
show(explain(DEMO_KA))

## 7. Your own input
Paste a JSON record below and run the cell.

In [ ]:
MY_INPUT = json.loads("""
{
  "language": "en",
  "gene": "",
  "variant": "",
  "gene_info": {},
  "clinvar": {"classification": "", "review_status": ""},
  "ai_estimate": {"label": "", "model": "Evo 2"},
  "context_facts": []
}
""")
show(explain(MY_INPUT))